In [3]:
#from google.colab import drive
#drive.mount('/content/drive')

In [4]:
from pathlib import Path
import pandas as pd
import os
import sklearn
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
import PIL
from PIL import Image
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from sklearn.model_selection import train_test_split

In [5]:
print("Pandas Version:", pd.__version__)
print("Scikit-Learn Version", sklearn.__version__)
print("Tensorflow Version", tf.__version__)
print("PIL Version:", PIL.__version__)

Pandas Version: 2.2.2
Scikit-Learn Version 1.6.1
Tensorflow Version 2.20.0
PIL Version: 11.3.0


In [6]:
#Current Working Path
try:
    BASE_DIR = Path(__file__).resolve().parent.parent
except NameError:
    BASE_DIR = Path.cwd()

#dataset root
DATA_DIR = Path("/content/drive/MyDrive/Datasets/Image Dataset/data")

print("Train exists:", os.path.exists(os.path.join(DATA_DIR, "train")))
print("Test exists:", os.path.exists(os.path.join(DATA_DIR, "test")))
print("Train csv exists:", os.path.exists(os.path.join(DATA_DIR, "Training_set.csv")))
print("Test csv exists:", os.path.exists(os.path.join(DATA_DIR, "Testing_set.csv")))

Train exists: True
Test exists: True
Train csv exists: True
Test csv exists: True


In [7]:
#load the training csv dataset & display the 1st 5 rows
train_df = pd.read_csv(DATA_DIR / "Training_set.csv")
train_df.head()

,filename,label
0,Image_1.jpg,SOUTHERN DOGFACE
1,Image_2.jpg,ADONIS
2,Image_3.jpg,BROWN SIPROETA
3,Image_4.jpg,MONARCH
4,Image_5.jpg,GREEN CELLED CATTLEHEART


In [8]:
train_df['image_path'] = train_df['filename'].apply(
    lambda x: os.path.join(DATA_DIR, "train", x)
)

In [9]:
train_df.sample(5)

,filename,label,image_path
5052,Image_5053.jpg,RED CRACKER,/content/drive/MyDrive/Datasets/Image Dataset/...
2302,Image_2303.jpg,PINE WHITE,/content/drive/MyDrive/Datasets/Image Dataset/...
325,Image_326.jpg,CABBAGE WHITE,/content/drive/MyDrive/Datasets/Image Dataset/...
1970,Image_1971.jpg,DANAID EGGFLY,/content/drive/MyDrive/Datasets/Image Dataset/...
1108,Image_1109.jpg,BROWN SIPROETA,/content/drive/MyDrive/Datasets/Image Dataset/...


In [10]:
#checking NAN values
train_df.isna().sum()

,0
filename,0
label,0
image_path,0


In [11]:
#checking duplicates
train_df.duplicated().sum()

np.int64(0)

In [12]:
# Check the distribution of classes in the target column
train_df['label'].value_counts()

,count
label,
MOURNING CLOAK,131
SLEEPY ORANGE,107
ATALA,100
BROWN SIPROETA,99
SCARCE SWALLOW,97
...,...
AMERICAN SNOOT,74
GOLD BANDED,73
MALACHITE,73


In [13]:
# Print the minimum and maximum class frequencies for label column
print(f"Minimum frequency: {train_df['label'].value_counts().min()}")
print(f"Maximum frequency: {train_df['label'].value_counts().max()}")

Minimum frequency: 71
Maximum frequency: 131


Sufficient images in each class; Ok for training

In [14]:
# Check for any missing images by verifying if the paths exist
missing_files = train_df[~train_df['image_path'].apply(os.path.exists)]
print("Missing Count:",len(missing_files))

Missing Count: 0


In [15]:
#label encoding for label column
label_en = LabelEncoder()
train_df['encoded'] = label_en.fit_transform(train_df['label'])

In [16]:
train_df[['label', 'encoded']].head()

,label,encoded
0,SOUTHERN DOGFACE,66
1,ADONIS,0
2,BROWN SIPROETA,12
3,MONARCH,44
4,GREEN CELLED CATTLEHEART,33


# Now Real EDA start

In [17]:
#File formate check
valid_ext = (".jpg", ".jpeg", ".png")
invalid_files = []

for path in train_df['image_path']:
    if not path.lower().endswith(valid_ext):
        invalid_files.append(path)

print("Invalid formate images:", len(invalid_files))

Invalid formate images: 0


In [18]:
#RGB/ corruted image check
corrupted = []
non_rgb = []

for path in train_df['image_path']:
    try:
        img = Image.open(path)

        #check RGB
        if img.mode != "RGB":
            non_rgb.append(path)

            img.verify()
    except:
        corrupted.append(path)

print("Corrupted Images:", len(corrupted))
print("Non RGB:", len(non_rgb))

Corrupted Images: 0
Non RGB: 0


In [19]:
#small size image checking
small_files = []
for path in train_df['image_path']:
    try:
        size_kb = os.path.getsize(path) / 1024
        if size_kb < 5:
            small_files.append(path)
    except:
        pass

print("Small Files (<5kb):", len(small_files))

Small Files (<5kb): 0


In [20]:
train_files = set(train_df['filename'])

print("Total train images:", len(train_files))
print("unique images:", len(set(train_files)))

Total train images: 6499
unique images: 6499


# Preprocessing

In [21]:
def load_image(image_path, label):

    #read image file
    img = tf.io.read_file(image_path)
    #decode image
    img = tf.image.decode_jpeg(img, channels=3)
    #Resize image
    img = tf.image.resize(img, (224, 224))
    #normalize
    img = img / 255

    return img, label

This function handles the image preprocessing pipeline for the TensorFlow:

1.Load & Decode:Reads raw image files from disk and converts them into 3-channel (RGB) tensors.
2.Resize: Standardizes all images to a uniform size of 244 X 244 pixels, as required by the model.
3.Normalize:Scales pixel values from [0, 255] down to [0, 1] to ensure faster and more stable model training.

In short, it transforms raw image files into optimized mathematical tensors ready for deep learning training.

In [22]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["encoded"].values)
)

train_ds = train_ds.map(load_image)

train_ds = train_ds.shuffle(1000)

train_ds = train_ds.batch(32)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

# Model training

In [23]:
base_model = MobileNetV2(
    weights= 'imagenet',
    include_top = False,
    input_shape = (224, 224, 3)
)

In [24]:
base_model.trainable = False

In [25]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(75, activation='softmax')
])

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 75)             │         9,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,431,627 (9.28 MB)

 Trainable params: 173,643 (678.29 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [27]:
model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 75)             │         9,675 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,431,627 (9.28 MB)

 Trainable params: 173,643 (678.29 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [28]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['encoded']
)

In [29]:
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["image_path"].values, train_df["encoded"].values)
)

train_ds = train_ds.map(load_image)
train_ds = train_ds.shuffle(1000)
train_ds = train_ds.batch(32)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

In [30]:
val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df['image_path'].values, val_df['encoded'].values)
)
val_ds = val_ds.map(load_image)
val_ds = val_ds.batch(32)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [31]:
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 318s 2s/step - accuracy: 0.0465 - loss: 4.2252 - val_accuracy: 0.1431 - val_loss: 3.9122
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 219s 1s/step - accuracy: 0.1735 - loss: 3.6724 - val_accuracy: 0.3423 - val_loss: 3.2818
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 208s 995ms/step - accuracy: 0.3193 - loss: 3.0761 - val_accuracy: 0.4862 - val_loss: 2.6734
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 193s 945ms/step - accuracy: 0.4210 - loss: 2.5429 - val_accuracy: 0.5862 - val_loss: 2.1984
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 156s 950ms/step - accuracy: 0.4976 - loss: 2.1890 - val_accuracy: 0.6623 - val_loss: 1.8557
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 205s 970ms/step - accuracy: 0.5770 - loss: 1.8593 - val_accuracy: 0.7069 - val_loss: 1.6019
Epoch 7/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 191s 905ms/step - accuracy: 0.6147 - loss: 1.6390 - val_accuracy: 0.7323 - val_loss: 1.4180
Epoch 8/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 204s 919ms/step - accuracy: 0.6449 - loss: